# レッスン 11 - エージェント間 (A2A) プロトコル


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2Aプロトコルとは何ですか？

The **Agent-to-Agent (A2A) protocol** は、AIエージェントが通信し、お互いを検出し、協力できるようにするオープンな標準であり、異なるフレームワーク上で構築されていたり、異なるサービスでホストされていたりしても機能します。

主要概念：

- **発見** - エージェントは自身の能力を説明する *エージェントカード* を公開し、
  他のエージェント（またはオーケストレーター）がタスクに適した専門家を簡単に見つけられるようにします。
- **メッセージパッシング** - エージェントは共通のプロトコルを介して構造化されたメッセージを交換するため、
  あるエージェントからのリクエストは、内部実装に関係なく、別のエージェントによって理解され、処理されます。

- **タスクライフサイクル** - A2A は *送信済み*、*作業中*、*完了*、失敗* などの
  *状態を定義し、オーケストレーターは委任されたタスクの進捗状況を完全に把握できます。

このレッスンでは、3人の専門旅行代理店をワークフローに接続することで、A2A（エージェント間）型のコラボレーションを
シミュレーションします。各代理店はそれぞれの専門知識を提供し、結果を次の代理店に渡します。


## 特化型旅行代理店の作成


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ワークフローによるマルチエージェント協調

3つのエージェントを、A2A メッセージパッシングを反映した逐次的なワークフローに接続します:

1. **CurrencyExchangeAgent** はユーザーのリクエストを受け取り、通貨に関するガイダンスを生成します。
2. **ActivityPlannerAgent** は強化されたコンテキストを受け取り、アクティビティの推奨を追加します。
3. **TravelManagerAgent** は両方の入力を統合して最終的な旅行概要を作成します。


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 本番環境における A2A の理解

本番環境では、A2A プロトコルにより強力なサービス間シナリオが実現します:

| 機能 | 説明 |
|---|---|
| **クロスフレームワーク相互運用** | あるフレームワークで構築されたエージェントは、他の任意のA2A準拠フレームワークで構築されたエージェントにタスクを委譲でき、真の組織間相互運用性を可能にします。 |
| **サービス境界** | エージェントは別々のマイクロサービス、クラウドリージョン、さらには異なる組織に存在しながらもシームレスに協調できます。 |
| **動的ディスカバリー** | オーケストレーターは実行時にAgent Card レジストリをクエリして、特定のサブタスクに最適なスペシャリストを見つけることができます。 |
| **ストリーミング & プッシュ通知** | A2A はリアルタイムの進行状況更新のために Server-Sent Events (SSE) をサポートし、長時間実行されるタスク向けにプッシュ通知を提供します。 |

上で構築したワークフローは、このパターンの簡略化されたプロセス内のバージョンです。実際の
デプロイメントでは、各エージェントが HTTP エンドポイントを公開し、Agent Card を発行し、通信
A2A JSON-RPC プロトコルを介して行われます。


## 要約

このレッスンで学んだこと:

1. **A2A プロトコルとは何か** — エージェント間のディスカバリ、メッセージング、
   およびタスク管理のためのオープン標準。
2. **特化したエージェントの作成方法** — 通貨交換エージェント、アクティビティプランナーエージェント、およびトラベルマネージャーオーケストレーター。
3. **エージェントをワークフローに組み込む方法** — `WorkflowBuilder` を使用してエージェント間の逐次的な
   メッセージ送受信をモデル化する。
4. **A2A が本番環境でどのように機能するか** — 動的ディスカバリとストリーミング更新により、フレームワーク横断・サービス横断のコラボレーションを実現する。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責事項：
本ドキュメントは AI 翻訳サービス「[Co-op Translator](https://github.com/Azure/co-op-translator)」を使用して翻訳されました。正確性には努めていますが、自動翻訳には誤りや不正確な箇所が含まれる可能性があることにご注意ください。原文の言語によるオリジナル文書を正式な情報源とみなしてください。重要な情報については、専門の翻訳者による人間翻訳を推奨します。本翻訳の利用に起因する誤解や解釈の相違について、当方は一切責任を負いません。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
